# **📘 Actividad 4. Selección y evaluación de modelos preentrenados**

**Materia:** Gestión de Proyectos de Inteligencia Artificial

**Proyecto:** Asistente inteligente para comprensión, orientación y cotización de seguros de vida mediante IA generativa.

**Notebook:** Evaluación comparativa con el dataset preparado y validado.

---

## **📝 1. Definición del caso de uso**

El proyecto contempla un asistente inteligente para apoyar la comprensión y orientación sobre seguros de vida.

En esta actividad se evalúa un componente de procesamiento de lenguaje natural que clasifica consultas en seis categorías: LIFE_INSURANCE, COVERAGE, QUOTE, PAYMENT, ENROLLMENT y CONTACT.

La clasificación permitirá orientar el procesamiento posterior de las consultas dentro del asistente.

### **🎯 1.1 Objetivo de la experimentación**

Comparar tres modelos preentrenados de Hugging Face mediante exactitud, precisión macro, exhaustividad macro, F1 macro y tiempos de ejecución.

Los modelos evaluados serán:

- MiniLMv2 multilingüe.
- mDeBERTa-v3-base.
- XLM-RoBERTa-large.

El criterio principal de selección será F1 macro. También se analizarán los resultados por categoría y la relación entre calidad de clasificación y tiempo de ejecución.

### **🎯 1.2 Origen y propósito del insumo**

Este notebook utiliza el archivo:

**INSUMOS_COLAB_2_COMPARACION_MODELOS.zip**

El insumo se generó en un primer notebook a partir del dataset **Bitext Insurance LLM Chatbot Training Dataset**, disponible en Hugging Face.

En ese notebook se seleccionaron las categorías pertinentes, se realizó el muestreo, se tradujeron las consultas al español y se revisaron las traducciones. También se sustituyeron los registros con coincidencias textuales detectadas, conservando los originales retirados para auditoría.

El dataset resultante contiene:

- **60 consultas de desarrollo:** 10 por categoría.
- **240 consultas de prueba:** 40 por categoría.
- **300 consultas en total**, con traducciones y categorías de referencia validadas.

Cargamos este insumo para reutilizar los datos preparados y concentrar este notebook en la evaluación de los modelos. No es necesario repetir el muestreo, la traducción ni los reemplazos.

El primer notebook se conserva como evidencia del procedimiento de preparación.

### **🎯 1.3 Condiciones de la evaluación**

Los tres modelos utilizarán las mismas consultas de prueba y la configuración fijada durante desarrollo.

No se modificarán las descripciones de las categorías a partir de los resultados de prueba. Las tablas y conclusiones se elaborarán con los resultados de una nueva evaluación cuando haya GPU, o con las evidencias
verificadas del primer Colab cuando no esté disponible.
En ambos casos se indicará su procedencia.

Se realizarán tres repeticiones por modelo para medir los tiempos y comprobar la estabilidad de las predicciones. Estas repeticiones utilizan el mismo dataset; no representan tres muestras independientes.

**Alcance:** Se evaluará la clasificación de consultas. Las traducciones no presentan coincidencias textuales según la normalización aplicada, aunque pueden persistir consultas semánticamente similares.

---

## **💻 2. Preparación del entorno**

Se preparará el entorno de Google Colab para cargar los modelos y ejecutar la evaluación.

Esta sección incluye la instalación de dependencias, la comprobación de la GPU y la autenticación en Hugging Face Hub.

### **🧠 2.1 Instalación de dependencias**

Antes de ejecutar el código, selecciona:

**Entorno de ejecución → Cambiar tipo de entorno de ejecución → Acelerador de hardware → GPU**

Se instalarán las bibliotecas necesarias para cargar los modelos, ejecutar la clasificación y calcular las métricas.

In [20]:
# ============================================================
# 2.1 Instalación de dependencias
# ============================================================

print("=" * 60)
print("📦 Instalando las dependencias para comparar los modelos...")
print("=" * 60)

# Instalamos las herramientas de inferencia y evaluación
%pip install -q transformers huggingface_hub sentencepiece scikit-learn pandas numpy protobuf

print("\nBibliotecas utilizadas:")
print("  • transformers: carga de modelos y pipeline de inferencia.")
print("  • huggingface_hub: acceso y autenticación en Hugging Face.")
print("  • sentencepiece y protobuf: soporte para tokenizadores.")
print("  • scikit-learn: cálculo de métricas de clasificación.")
print("  • pandas y numpy: organización de datos y resultados.")

print("\nSiguiente paso: comprobar la GPU y registrar el entorno.")

print("=" * 60)
print("✅ Instalación de dependencias completada.")
print("=" * 60)

📦 Instalando las dependencias para comparar los modelos...

Bibliotecas utilizadas:
  • transformers: carga de modelos y pipeline de inferencia.
  • huggingface_hub: acceso y autenticación en Hugging Face.
  • sentencepiece y protobuf: soporte para tokenizadores.
  • scikit-learn: cálculo de métricas de clasificación.
  • pandas y numpy: organización de datos y resultados.

Siguiente paso: comprobar la GPU y registrar el entorno.
✅ Instalación de dependencias completada.


### **🧠 2.2 Comprobación de GPU y registro del entorno**

Se registrarán las versiones del entorno y se comprobará
la disponibilidad de GPU.

El notebook tendrá dos modos de trabajo:

- **Nueva evaluación:** si hay GPU, se ejecutarán los tres modelos.
- **Recuperación de evidencias:** si no hay GPU, se cargarán los
  resultados obtenidos en el primer Colab.

Las tablas y respaldos indicarán la procedencia de los resultados.
En el modo de recuperación, el entorno actual no se presentará
como el entorno donde se realizó la evaluación anterior.

La memoria total de la GPU corresponde a la capacidad del
dispositivo, no al consumo de cada modelo.

In [21]:
# ============================================================
# 2.2 Comprobación de GPU y registro del entorno
# ============================================================

import platform
from datetime import datetime, timezone
from importlib.metadata import version

import pandas as pd
import torch

print("=" * 60)
print("🔎 Comprobando los recursos y registrando el entorno...")
print("=" * 60)

# Registramos las versiones del entorno actual
bibliotecas = [
    "torch",
    "transformers",
    "huggingface_hub",
    "sentencepiece",
    "scikit-learn",
    "pandas",
    "numpy",
    "protobuf"
]

versiones_bibliotecas = {
    biblioteca: version(biblioteca)
    for biblioteca in bibliotecas
}

# Determinamos el modo sin detener el notebook si falta GPU
gpu_disponible = torch.cuda.is_available()

modo_evaluacion = (
    "nueva_evaluacion"
    if gpu_disponible
    else "recuperacion_evidencias"
)

dispositivo = 0 if gpu_disponible else None

registro_entorno = {
    "fecha_registro_utc": datetime.now(timezone.utc).isoformat(),
    "python": platform.python_version(),
    "sistema": platform.platform(),
    "gpu_disponible": gpu_disponible,
    "gpu": None,
    "memoria_total_gpu_gib": None,
    "cuda_pytorch": torch.version.cuda,
    "bibliotecas": versiones_bibliotecas,
    "modo_evaluacion": modo_evaluacion
}

if gpu_disponible:
    propiedades_gpu = torch.cuda.get_device_properties(dispositivo)

    registro_entorno["gpu"] = propiedades_gpu.name
    registro_entorno["memoria_total_gpu_gib"] = round(
        propiedades_gpu.total_memory / (1024 ** 3), 2
    )

    print("\nGPU detectada:", registro_entorno["gpu"])
    print(
        "Memoria total de la GPU:",
        f"{registro_entorno['memoria_total_gpu_gib']:.2f} GiB"
    )
    print("Modo seleccionado: NUEVA EVALUACIÓN.")
    print("Se ejecutarán los tres modelos en esta sesión.")

else:
    print("\nNo hay GPU disponible en esta sesión.")
    print("Modo seleccionado: RECUPERACIÓN DE EVIDENCIAS.")
    print("No se ejecutarán los modelos en CPU.")
    print("Se solicitará posteriormente este respaldo:")
    print("evidencias_reporte_20261010T200713130418Z.zip")
    print(
        "Los resultados recuperados se identificarán como "
        "procedentes del primer Colab."
    )

display(
    pd.DataFrame(
        versiones_bibliotecas.items(),
        columns=["Biblioteca", "Versión instalada"]
    )
)

print("\nVersión de Python:", registro_entorno["python"])
print("Registro del entorno actual conservado en: registro_entorno")
print("Modo de trabajo conservado en: modo_evaluacion")

print("\nEste paso no ejecuta ni recupera resultados.")
print("Los datos y la configuración se cargarán en los pasos siguientes.")

print("=" * 60)
print("✅ Entorno registrado y modo de trabajo establecido.")
print("=" * 60)

🔎 Comprobando los recursos y registrando el entorno...

No hay GPU disponible en esta sesión.
Modo seleccionado: RECUPERACIÓN DE EVIDENCIAS.
No se ejecutarán los modelos en CPU.
Se solicitará posteriormente este respaldo:
evidencias_reporte_20261010T200713130418Z.zip
Los resultados recuperados se identificarán como procedentes del primer Colab.


,Biblioteca,Versión instalada
0,torch,2.11.0+cpu
1,transformers,5.18.0
2,huggingface_hub,1.33.0
3,sentencepiece,0.2.2
4,scikit-learn,1.6.1
5,pandas,2.2.3
6,numpy,2.1.3
7,protobuf,6.33.6



Versión de Python: 3.13.16
Registro del entorno actual conservado en: registro_entorno
Modo de trabajo conservado en: modo_evaluacion

Este paso no ejecuta ni recupera resultados.
Los datos y la configuración se cargarán en los pasos siguientes.
✅ Entorno registrado y modo de trabajo establecido.


### **🧠 2.3 Autenticación en Hugging Face Hub**

Se iniciará sesión en Hugging Face para acceder a los repositorios
de los modelos.

El token se introducirá mediante un campo oculto y no se escribirá
en el código ni se incluirá en los respaldos.

Esta etapa puede ejecutarse sin GPU. Todavía no se descargarán
los modelos ni se realizará inferencia.

In [22]:
# ============================================================
# 2.3 Autenticación en Hugging Face Hub
# ============================================================

from getpass import getpass
from huggingface_hub import login, whoami

print("=" * 60)
print("🔐 Preparando la autenticación en Hugging Face...")
print("=" * 60)

# Solicitamos el token sin mostrarlo en pantalla
token_hf = getpass("Introduce tu token de Hugging Face: ").strip()

if not token_hf:
    raise ValueError(
        "No se introdujo un token. Vuelve a ejecutar esta celda."
    )

# Autenticamos y eliminamos la variable que contiene el token
try:
    login(token=token_hf, add_to_git_credential=False)
    usuario_hf = whoami()["name"]
finally:
    del token_hf

print("\nUsuario autenticado:", usuario_hf)
print("No se han cargado modelos ni ejecutado inferencia.")
print(
    "Siguiente paso: cargar "
    "INSUMOS_COLAB_2_COMPARACION_MODELOS.zip."
)

print("=" * 60)
print("✅ Autenticación en Hugging Face completada.")
print("=" * 60)

🔐 Preparando la autenticación en Hugging Face...
Introduce tu token de Hugging Face: ··········

Usuario autenticado: mauroene
No se han cargado modelos ni ejecutado inferencia.
Siguiente paso: cargar INSUMOS_COLAB_2_COMPARACION_MODELOS.zip.
✅ Autenticación en Hugging Face completada.


## **📦 3. Carga y verificación de los insumos finales**

Se cargará el paquete generado en el primer Colab, que contiene
los datasets traducidos y validados, la configuración experimental
y los archivos de trazabilidad y auditoría.

Estos insumos permiten continuar sin repetir la preparación
del dataset.

### **🧠 3.1 Carga del paquete de entrada**

Selecciona desde tu computadora el archivo:

**INSUMOS_COLAB_2_COMPARACION_MODELOS.zip**

Se comprobará que el ZIP pueda leerse y que incluya los ocho
archivos esperados. La verificación del manifiesto y la carga
de los datasets se realizarán en el siguiente paso.

**Estado:** No se ejecutará inferencia.

In [24]:
# ============================================================
# 3.1 Carga del paquete de entrada
# ============================================================

from pathlib import Path
from zipfile import ZipFile, is_zipfile
from google.colab import files

print("=" * 60)
print("📂 Cargando el insumo generado en el primer Colab...")
print("=" * 60)

print("\nSelecciona únicamente este archivo:")
print("INSUMOS_COLAB_2_COMPARACION_MODELOS.zip")
print("Se aceptará el nombre con sufijo que Colab pueda asignarle.")

# Abrimos el selector de archivos
archivos_subidos = files.upload()

if len(archivos_subidos) != 1:
    raise ValueError(
        "Selecciona un solo archivo ZIP y vuelve a ejecutar la celda."
    )

# Recuperamos el nombre real utilizado por Colab
nombre_cargado = next(iter(archivos_subidos))
ruta_zip_insumos = Path.cwd() / nombre_cargado

if not is_zipfile(ruta_zip_insumos):
    raise ValueError("El archivo seleccionado no es un ZIP válido.")

archivos_esperados = {
    "dataset_desarrollo.csv",
    "dataset_prueba.csv",
    "configuracion_para_prueba.json",
    "dataset_completo_trazabilidad.csv",
    "registros_retirados.csv",
    "distribucion_dataset.csv",
    "LEEME_COLAB_2.txt",
    "manifest_insumos.json"
}

# Identificamos el paquete por su contenido
with ZipFile(ruta_zip_insumos, "r") as paquete:
    archivo_danado = paquete.testzip()

    if archivo_danado is not None:
        raise ValueError(
            f"Se detectó un archivo dañado: {archivo_danado}."
        )

    nombres_archivos = paquete.namelist()

    if len(nombres_archivos) != len(set(nombres_archivos)):
        raise ValueError("El ZIP contiene nombres duplicados.")

    if set(nombres_archivos) != archivos_esperados:
        raise ValueError(
            "El ZIP no contiene los ocho archivos esperados. "
            "Selecciona el paquete de INSUMOS_COLAB_2."
        )

print("\n📍 ZIP cargado en:", ruta_zip_insumos)

print("\n📋 Archivos incluidos:")
for nombre in sorted(nombres_archivos):
    print("  •", nombre)

print("\nEl ZIP puede leerse y contiene los ocho archivos requeridos.")
print("Siguiente paso: ejecutar 3.2 para verificar el manifiesto.")
print("No se han cargado modelos ni ejecutado inferencia.")

print("=" * 60)
print("✅ Paquete de entrada cargado y contenido comprobado.")
print("=" * 60)

📂 Cargando el insumo generado en el primer Colab...

Selecciona únicamente este archivo:
INSUMOS_COLAB_2_COMPARACION_MODELOS.zip
Se aceptará el nombre con sufijo que Colab pueda asignarle.


Saving INSUMOS_COLAB_2_COMPARACION_MODELOS.zip to INSUMOS_COLAB_2_COMPARACION_MODELOS (3).zip

📍 ZIP cargado en: /content/INSUMOS_COLAB_2_COMPARACION_MODELOS (3).zip

📋 Archivos incluidos:
  • LEEME_COLAB_2.txt
  • configuracion_para_prueba.json
  • dataset_completo_trazabilidad.csv
  • dataset_desarrollo.csv
  • dataset_prueba.csv
  • distribucion_dataset.csv
  • manifest_insumos.json
  • registros_retirados.csv

El ZIP puede leerse y contiene los ocho archivos requeridos.
Siguiente paso: ejecutar 3.2 para verificar el manifiesto.
No se han cargado modelos ni ejecutado inferencia.
✅ Paquete de entrada cargado y contenido comprobado.


### **🧠 3.2 Verificación de integridad y extracción de insumos**

Se comparará la huella SHA-256 de los seis archivos de datos
y configuración con la registrada en manifest_insumos.json.

Esta comprobación permite detectar cambios respecto de los
archivos exportados en el primer Colab; no sustituye la revisión
semántica realizada durante la preparación.

Después se extraerán los ocho archivos del paquete en una
carpeta nueva, conservando cualquier extracción anterior.

**Estado:** Verificación y extracción. No se ejecutará inferencia.

In [25]:
# ============================================================
# 3.2 Verificación de integridad y extracción de insumos
# ============================================================

import json
import hashlib
from datetime import datetime, timezone
from pathlib import Path
from zipfile import ZipFile

print("=" * 60)
print("🔎 Verificando la integridad de los insumos...")
print("=" * 60)

# Definimos los archivos que deben tener una huella registrada
archivos_con_huella = {
    "dataset_desarrollo.csv",
    "dataset_prueba.csv",
    "configuracion_para_prueba.json",
    "dataset_completo_trazabilidad.csv",
    "registros_retirados.csv",
    "distribucion_dataset.csv"
}

archivos_paquete = archivos_con_huella | {
    "LEEME_COLAB_2.txt",
    "manifest_insumos.json"
}

with ZipFile(ruta_zip_insumos, "r") as paquete:
    nombres = paquete.namelist()

    # Solo aceptamos los ocho archivos esperados en la raíz
    if (
        len(nombres) != len(archivos_paquete)
        or set(nombres) != archivos_paquete
    ):
        raise ValueError(
            "El contenido del ZIP no coincide con el paquete esperado."
        )

    if paquete.testzip() is not None:
        raise ValueError("El ZIP contiene un archivo dañado.")

    manifest_insumos = json.loads(
        paquete.read("manifest_insumos.json").decode("utf-8")
    )

    if set(manifest_insumos["archivos"]) != archivos_con_huella:
        raise ValueError(
            "El manifiesto no registra los seis insumos esperados."
        )

    # Verificamos todos los archivos antes de extraerlos
    verificacion_insumos = []

    for nombre in sorted(archivos_con_huella):
        huella_actual = hashlib.sha256(
            paquete.read(nombre)
        ).hexdigest()

        huella_esperada = (
            manifest_insumos["archivos"][nombre]["sha256"]
        )

        if huella_actual != huella_esperada:
            raise ValueError(
                f"La huella de {nombre} no coincide con el manifiesto."
            )

        verificacion_insumos.append({
            "Archivo": nombre,
            "Verificación SHA-256": "Correcta"
        })

    # Creamos una carpeta nueva para cada extracción
    marca_extraccion = datetime.now(timezone.utc).strftime(
        "%Y%m%dT%H%M%S%fZ"
    )

    carpeta_insumos = (
        Path("/content/insumos_colab_2") / marca_extraccion
    )
    carpeta_insumos.mkdir(parents=True, exist_ok=False)

    # Extraemos únicamente los nombres comprobados
    for nombre in sorted(archivos_paquete):
        (carpeta_insumos / nombre).write_bytes(
            paquete.read(nombre)
        )

display(pd.DataFrame(verificacion_insumos))

print(
    "\nVersión del dataset:",
    manifest_insumos["version_dataset_utc"]
)
print(
    "Consultas de desarrollo declaradas:",
    manifest_insumos["consultas_desarrollo"]
)
print(
    "Consultas de prueba declaradas:",
    manifest_insumos["consultas_prueba"]
)

print("\n📁 Carpeta de los insumos extraídos:")
print(carpeta_insumos)

print("\n📋 Uso de los archivos:")
print("  • dataset_desarrollo.csv: conjunto de desarrollo.")
print("  • dataset_prueba.csv: conjunto para evaluar los modelos.")
print("  • configuracion_para_prueba.json: modelos y parámetros fijados.")
print("  • dataset_completo_trazabilidad.csv: originales y traducciones.")
print("  • registros_retirados.csv: auditoría de los reemplazos.")
print("  • distribucion_dataset.csv: distribución por conjunto y categoría.")
print("  • LEEME_COLAB_2.txt: guía del paquete.")
print("  • manifest_insumos.json: versión y huellas de los seis insumos.")

print("\nSiguiente paso: cargar y comprobar los datasets.")
print("No se han cargado modelos ni ejecutado inferencia.")

print("=" * 60)
print("✅ Integridad verificada e insumos extraídos.")
print("=" * 60)

🔎 Verificando la integridad de los insumos...


,Archivo,Verificación SHA-256
0,configuracion_para_prueba.json,Correcta
1,dataset_completo_trazabilidad.csv,Correcta
2,dataset_desarrollo.csv,Correcta
3,dataset_prueba.csv,Correcta
4,distribucion_dataset.csv,Correcta
5,registros_retirados.csv,Correcta



Versión del dataset: 20261010T195212647311Z
Consultas de desarrollo declaradas: 60
Consultas de prueba declaradas: 240

📁 Carpeta de los insumos extraídos:
/content/insumos_colab_2/20261010T213025424734Z

📋 Uso de los archivos:
  • dataset_desarrollo.csv: conjunto de desarrollo.
  • dataset_prueba.csv: conjunto para evaluar los modelos.
  • configuracion_para_prueba.json: modelos y parámetros fijados.
  • dataset_completo_trazabilidad.csv: originales y traducciones.
  • registros_retirados.csv: auditoría de los reemplazos.
  • distribucion_dataset.csv: distribución por conjunto y categoría.
  • LEEME_COLAB_2.txt: guía del paquete.
  • manifest_insumos.json: versión y huellas de los seis insumos.

Siguiente paso: cargar y comprobar los datasets.
No se han cargado modelos ni ejecutado inferencia.
✅ Integridad verificada e insumos extraídos.


### **🧠 3.3 Carga y comprobación de los datasets**

Se cargarán los conjuntos de desarrollo y prueba, conservando
la separación establecida en el primer Colab.

Se comprobará que contengan 60 y 240 consultas respectivamente,
con 10 consultas de desarrollo y 40 de prueba por categoría.

También se verificarán los campos requeridos, los identificadores,
las traducciones completas y las coincidencias textuales entre
ambos conjuntos.

Estas comprobaciones no modifican los datos ni sustituyen
la validación semántica documentada en el primer notebook.

**Estado:** Datos preparados para evaluación; inferencia pendiente.

In [26]:
# ============================================================
# 3.3 Carga y comprobación de los datasets
# ============================================================

import unicodedata
import pandas as pd

print("=" * 60)
print("📋 Cargando y comprobando los datasets...")
print("=" * 60)

# Recuperamos las categorías declaradas en el manifiesto
categorias = list(manifest_insumos["categorias"])

if len(categorias) != 6 or len(set(categorias)) != 6:
    raise ValueError("El manifiesto debe declarar seis categorías únicas.")

# Cargamos los datos sin convertir campos vacíos en valores NaN
df_desarrollo = pd.read_csv(
    carpeta_insumos / "dataset_desarrollo.csv",
    encoding="utf-8-sig",
    dtype=str,
    keep_default_na=False
)

df_prueba = pd.read_csv(
    carpeta_insumos / "dataset_prueba.csv",
    encoding="utf-8-sig",
    dtype=str,
    keep_default_na=False
)

campos_requeridos = [
    "id_consulta",
    "consulta_es",
    "categoria_referencia"
]

# Normalizamos únicamente para comprobar coincidencias
# Los textos guardados en los datasets permanecen intactos
def normalizar_comprobacion(texto):
    return " ".join(
        unicodedata.normalize("NFKC", texto).casefold().split()
    )

# Verificamos cada conjunto
for nombre, datos, total_esperado, cantidad_categoria in [
    ("desarrollo", df_desarrollo, 60, 10),
    ("prueba", df_prueba, 240, 40)
]:
    faltantes = set(campos_requeridos) - set(datos.columns)

    if faltantes:
        raise ValueError(
            f"Faltan campos en {nombre}: {sorted(faltantes)}."
        )

    if len(datos) != total_esperado:
        raise ValueError(
            f"{nombre}: se esperaban {total_esperado} consultas "
            f"y se encontraron {len(datos)}."
        )

    for campo in campos_requeridos:
        if datos[campo].str.strip().eq("").any():
            raise ValueError(
                f"Se encontraron valores vacíos en {nombre}: {campo}."
            )

    if datos["id_consulta"].duplicated().any():
        raise ValueError(f"Hay identificadores repetidos en {nombre}.")

    if not datos["categoria_referencia"].isin(categorias).all():
        raise ValueError(f"Hay categorías no previstas en {nombre}.")

    cantidades = (
        datos["categoria_referencia"]
        .value_counts()
        .reindex(categorias, fill_value=0)
    )

    if not cantidades.eq(cantidad_categoria).all():
        raise ValueError(
            f"La distribución por categoría de {nombre} "
            "no coincide con lo previsto."
        )

# Comprobamos la separación entre desarrollo y prueba
ids_compartidos = (
    set(df_desarrollo["id_consulta"])
    & set(df_prueba["id_consulta"])
)

if ids_compartidos:
    raise ValueError(
        "Desarrollo y prueba comparten identificadores: "
        + ", ".join(sorted(ids_compartidos))
    )

textos_completos = pd.concat(
    [
        df_desarrollo["consulta_es"],
        df_prueba["consulta_es"]
    ],
    ignore_index=True
).map(normalizar_comprobacion)

if textos_completos.eq("").any():
    raise ValueError("Hay traducciones vacías después de normalizar.")

if textos_completos.duplicated().any():
    raise ValueError(
        "Se detectaron traducciones repetidas dentro de los "
        "conjuntos o entre desarrollo y prueba."
    )

# Comprobamos que las cantidades coincidan con el manifiesto
if (
    len(df_desarrollo) != manifest_insumos["consultas_desarrollo"]
    or len(df_prueba) != manifest_insumos["consultas_prueba"]
):
    raise ValueError("Las cantidades no coinciden con el manifiesto.")

# Presentamos la distribución recuperada
resumen_dataset = pd.DataFrame({
    "Categoría": categorias,
    "Desarrollo": [
        df_desarrollo["categoria_referencia"].eq(c).sum()
        for c in categorias
    ],
    "Prueba": [
        df_prueba["categoria_referencia"].eq(c).sum()
        for c in categorias
    ]
})

resumen_dataset["Total"] = (
    resumen_dataset["Desarrollo"] + resumen_dataset["Prueba"]
)

display(resumen_dataset)

print("\nConsultas de desarrollo:", len(df_desarrollo))
print("Consultas de prueba:", len(df_prueba))
print("Consultas totales:", len(textos_completos))
print("Identificadores compartidos entre conjuntos: 0")
print("Traducciones vacías: 0")
print("Traducciones repetidas según la normalización aplicada: 0")

print("\n📋 Campos de los datasets:")
print("  • id_consulta: identificador del registro.")
print("  • consulta_es: texto en español que recibirá el modelo.")
print("  • categoria_referencia: etiqueta para calcular las métricas.")

print("\nVariables disponibles:")
print("  • df_desarrollo: 60 consultas de desarrollo.")
print("  • df_prueba: 240 consultas para la evaluación.")
print("  • categorias: orden de las seis categorías.")
print("  • resumen_dataset: distribución de los conjuntos.")

print("\nSiguiente paso: recuperar los modelos y la configuración fijada.")
print("No se ha ejecutado inferencia.")

print("=" * 60)
print("✅ Datasets cargados y comprobaciones completadas.")
print("=" * 60)

📋 Cargando y comprobando los datasets...


,Categoría,Desarrollo,Prueba,Total
0,LIFE_INSURANCE,10,40,50
1,COVERAGE,10,40,50
2,QUOTE,10,40,50
3,PAYMENT,10,40,50
4,ENROLLMENT,10,40,50
5,CONTACT,10,40,50



Consultas de desarrollo: 60
Consultas de prueba: 240
Consultas totales: 300
Identificadores compartidos entre conjuntos: 0
Traducciones vacías: 0
Traducciones repetidas según la normalización aplicada: 0

📋 Campos de los datasets:
  • id_consulta: identificador del registro.
  • consulta_es: texto en español que recibirá el modelo.
  • categoria_referencia: etiqueta para calcular las métricas.

Variables disponibles:
  • df_desarrollo: 60 consultas de desarrollo.
  • df_prueba: 240 consultas para la evaluación.
  • categorias: orden de las seis categorías.
  • resumen_dataset: distribución de los conjuntos.

Siguiente paso: recuperar los modelos y la configuración fijada.
No se ha ejecutado inferencia.
✅ Datasets cargados y comprobaciones completadas.


## **⚙️ 4. Modelos y configuración experimental**

Se compararán tres modelos preentrenados mediante clasificación
zero-shot. Cada modelo evaluará la relación entre una consulta
y las descripciones de las seis categorías.

Los identificadores de los modelos y los parámetros se recuperarán
del paquete del primer Colab para conservar la configuración
fijada durante desarrollo.

### **🧠 4.1 Recuperación de la configuración**

Se cargarán los repositorios de los tres modelos, las descripciones
de las categorías y los parámetros de inferencia.

La evaluación utilizará una sola categoría por consulta, tamaño
de lote de uno, precisión float32 y tres repeticiones por modelo.

**Criterio principal:** F1 macro.

**Estado:** Configuración recuperada; carga de modelos e inferencia
pendientes de disponibilidad de GPU.

In [28]:
# ============================================================
# 4.1 Recuperación de la configuración
# ============================================================

import json
import pandas as pd

print("=" * 60)
print("⚙️ Recuperando los modelos y la configuración experimental...")
print("=" * 60)

# Cargamos la configuración verificada en el paso 3.2
with (carpeta_insumos / "configuracion_para_prueba.json").open(
    encoding="utf-8"
) as archivo:
    registro_configuracion = json.load(archivo)

claves_requeridas = {
    "modelos",
    "configuracion_prueba",
    "decision_desarrollo"
}

if not claves_requeridas.issubset(registro_configuracion):
    raise ValueError(
        "El archivo no contiene todos los campos de configuración."
    )

modelos_comparacion = registro_configuracion["modelos"]
configuracion_prueba = registro_configuracion["configuracion_prueba"]
decision_desarrollo = registro_configuracion["decision_desarrollo"]

# Comprobamos los identificadores esperados
modelos_esperados = {
    "MiniLMv2 multilingüe":
        "MoritzLaurer/multilingual-MiniLMv2-L6-mnli-xnli",
    "mDeBERTa-v3-base":
        "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli",
    "XLM-RoBERTa-large":
        "joeddav/xlm-roberta-large-xnli"
}

if modelos_comparacion != modelos_esperados:
    raise ValueError(
        "Los modelos guardados no coinciden con los tres "
        "repositorios previstos."
    )

# Recuperamos las descripciones sin modificarlas
descripciones = configuracion_prueba["descripciones"]

if set(descripciones) != set(categorias):
    raise ValueError(
        "Las descripciones no corresponden a las seis categorías."
    )

if not all(
    isinstance(texto, str) and texto.strip()
    for texto in descripciones.values()
):
    raise ValueError("Hay descripciones vacías o inválidas.")

# Verificamos el protocolo fijado en el primer Colab
parametros_esperados = {
    "plantilla": "Esta consulta trata sobre {}.",
    "multi_label": False,
    "batch_size": 1,
    "repeticiones": 3,
    "precision": "float32"
}

for parametro, valor_esperado in parametros_esperados.items():
    if configuracion_prueba.get(parametro) != valor_esperado:
        raise ValueError(
            f"El parámetro {parametro} no coincide con "
            "la configuración fijada."
        )

# Conservamos el mismo orden para categorías y descripciones
etiquetas_candidatas = [
    descripciones[categoria]
    for categoria in categorias
]

if len(set(etiquetas_candidatas)) != len(categorias):
    raise ValueError("Las descripciones de las categorías se repiten.")

descripcion_a_categoria = {
    descripciones[categoria]: categoria
    for categoria in categorias
}

print("\n📋 Modelos que se evaluarán:")
display(
    pd.DataFrame(
        modelos_comparacion.items(),
        columns=["Modelo", "Repositorio en Hugging Face"]
    )
)

print("\n📋 Descripciones recuperadas:")
display(
    pd.DataFrame({
        "Categoría": categorias,
        "Descripción": etiquetas_candidatas
    })
)

print("\n📋 Parámetros de evaluación:")
for parametro in parametros_esperados:
    print(f"  • {parametro}: {configuracion_prueba[parametro]}")

print("\nConsultas de prueba:", len(df_prueba))
print("Criterio principal de selección: F1 macro.")
print("La decisión previa de desarrollo se conserva como antecedente.")
print("La selección de esta ejecución se registrará después de evaluar.")

print("\n📍 Configuración recuperada desde:")
print(carpeta_insumos / "configuracion_para_prueba.json")

print("\nNo se han descargado modelos ni ejecutado inferencia.")
print("La evaluación con GPU permanece pendiente.")

print("=" * 60)
print("✅ Modelos y configuración experimental recuperados.")
print("=" * 60)

⚙️ Recuperando los modelos y la configuración experimental...

📋 Modelos que se evaluarán:


,Modelo,Repositorio en Hugging Face
0,MiniLMv2 multilingüe,MoritzLaurer/multilingual-MiniLMv2-L6-mnli-xnli
1,mDeBERTa-v3-base,MoritzLaurer/mDeBERTa-v3-base-mnli-xnli
2,XLM-RoBERTa-large,joeddav/xlm-roberta-large-xnli



📋 Descripciones recuperadas:


,Categoría,Descripción
0,LIFE_INSURANCE,información general o consulta de información ...
1,COVERAGE,"consulta de coberturas, exclusiones o modifica..."
2,QUOTE,"consulta de tarifas, estimación de costo o sol..."
3,PAYMENT,"realización y consulta de pagos del seguro, pr..."
4,ENROLLMENT,"comparación, contratación, renovación o cancel..."
5,CONTACT,"contacto con atención al cliente, un represent..."



📋 Parámetros de evaluación:
  • plantilla: Esta consulta trata sobre {}.
  • multi_label: False
  • batch_size: 1
  • repeticiones: 3
  • precision: float32

Consultas de prueba: 240
Criterio principal de selección: F1 macro.
La decisión previa de desarrollo se conserva como antecedente.
La selección de esta ejecución se registrará después de evaluar.

📍 Configuración recuperada desde:
/content/insumos_colab_2/20261010T213025424734Z/configuracion_para_prueba.json

No se han descargado modelos ni ejecutado inferencia.
La evaluación con GPU permanece pendiente.
✅ Modelos y configuración experimental recuperados.


### **🧠 4.2 Protocolo de evaluación y criterio de selección**

Los tres modelos se compararán sobre las mismas 240 consultas
de prueba, con las descripciones fijadas durante desarrollo.

En modo de nueva evaluación se realizará:

1. Carga de cada modelo y medición separada del tiempo de carga.
2. Calentamiento con dos consultas de desarrollo.
3. Tres repeticiones de inferencia sobre prueba.
4. Comprobación de estabilidad de las etiquetas.
5. Cálculo de métricas con la primera repetición.

En modo de recuperación se utilizarán los resultados verificados
del primer Colab. Aquella ejecución realizó el calentamiento con
las primeras dos consultas de prueba, fuera del tiempo medido.
Esta diferencia se conservará documentada al comparar tiempos
entre ejecuciones.

Se presentarán exactitud, precisión macro, exhaustividad macro,
F1 macro, resultados por categoría, errores y tiempos.

El tiempo medio por consulta corresponde al tiempo medio total
dividido entre las 240 consultas; no es una medición aislada
de latencia.

**Criterio principal:** Mayor F1 macro sobre prueba.

**Limitaciones:** Las repeticiones utilizan las mismas consultas
y no constituyen muestras independientes. Los puntajes del
modelo no son probabilidades calibradas.

**Procedencia:** Nueva ejecución con GPU o evaluación anterior
recuperada, según el modo registrado en el notebook.

### **📦 4.3 Carga condicional de evidencias anteriores**

El notebook utilizará el modo establecido en 2.2:

- **Nueva evaluación:** continuará hacia la ejecución de los modelos.
- **Recuperación de evidencias:** solicitará el respaldo de los
  resultados obtenidos en el primer Colab.

El respaldo de evidencias contiene métricas, predicciones, errores,
matrices de confusión, tiempos y el registro de la evaluación
realizada en Tesla T4.

Antes de utilizar esos resultados, se comprobará su integridad
y su correspondencia con el dataset y la configuración cargados.

**Importante:** Cargar el respaldo no equivale a ejecutar los modelos
en esta sesión.

In [29]:
# ============================================================
# 4.3 Carga condicional de evidencias anteriores
# ============================================================

from pathlib import Path
from zipfile import ZipFile, is_zipfile
from google.colab import files

print("=" * 60)
print("📦 Preparando la fuente de resultados...")
print("=" * 60)

ruta_zip_evidencias = None
evidencias_verificadas = False

if modo_evaluacion == "nueva_evaluacion":
    print("\nModo: NUEVA EVALUACIÓN.")
    print("No es necesario subir el respaldo de resultados.")
    print("Se ejecutarán los modelos en la etapa de evaluación.")

elif modo_evaluacion == "recuperacion_evidencias":
    print("\nModo: RECUPERACIÓN DE EVIDENCIAS.")
    print("Selecciona únicamente este archivo:")
    print("evidencias_reporte_20261010T200713130418Z.zip")
    print("Se aceptarán los sufijos que Colab agregue al nombre.")

    archivos_evidencias = files.upload()

    if len(archivos_evidencias) != 1:
        raise ValueError(
            "Selecciona un solo ZIP de evidencias."
        )

    # Utilizamos el nombre real asignado por Colab
    nombre_cargado = next(iter(archivos_evidencias))
    ruta_zip_evidencias = Path.cwd() / nombre_cargado

    if not is_zipfile(ruta_zip_evidencias):
        raise ValueError("El archivo seleccionado no es un ZIP válido.")

    archivos_requeridos = {
        "paquete_final/manifest_paquete.json",
        "paquete_final/dataset/dataset_prueba.csv",
        "paquete_final/dataset/configuracion_para_prueba.json",
        "paquete_final/resultados_prueba/registro_evaluacion.json",
        "paquete_final/resultados_prueba/comparacion_prueba.csv",
        "paquete_final/resultados_prueba/dataset_prueba_utilizado.csv"
    }

    # Identificamos el respaldo por su contenido
    with ZipFile(ruta_zip_evidencias, "r") as paquete:
        if paquete.testzip() is not None:
            raise ValueError("El ZIP contiene un archivo dañado.")

        nombres = paquete.namelist()

        if len(nombres) != len(set(nombres)):
            raise ValueError("El ZIP contiene nombres duplicados.")

        faltantes = archivos_requeridos - set(nombres)

        if faltantes:
            raise ValueError(
                "Este ZIP no contiene las evidencias requeridas. "
                "Selecciona evidencias_reporte_20261010T200713130418Z.zip. "
                "Archivos faltantes: "
                + ", ".join(sorted(faltantes))
            )

    print("\n📍 Respaldo cargado en:")
    print(ruta_zip_evidencias)
    print("\nLos archivos principales están presentes.")
    print("Siguiente paso: verificar integridad y correspondencia.")
    print("Todavía no se han recuperado métricas para la comparación.")

else:
    raise ValueError("Modo desconocido. Comprueba la celda 2.2.")

print("=" * 60)
print("✅ Preparación de la fuente de resultados completada.")
print("=" * 60)

📦 Preparando la fuente de resultados...

Modo: RECUPERACIÓN DE EVIDENCIAS.
Selecciona únicamente este archivo:
evidencias_reporte_20261010T200713130418Z.zip
Se aceptarán los sufijos que Colab agregue al nombre.


Saving evidencias_reporte_20261010T200713130418Z.zip to evidencias_reporte_20261010T200713130418Z.zip

📍 Respaldo cargado en:
/content/evidencias_reporte_20261010T200713130418Z.zip

Los archivos principales están presentes.
Siguiente paso: verificar integridad y correspondencia.
Todavía no se han recuperado métricas para la comparación.
✅ Preparación de la fuente de resultados completada.


### **🧠 4.4 Verificación de las evidencias recuperadas**

En modo de recuperación se comprobarán las huellas SHA-256
del paquete final y su correspondencia con los insumos actuales:

- Mismo dataset y versión.
- Mismos tres modelos.
- Mismas descripciones y parámetros de evaluación.
- Mismas 240 consultas utilizadas en la evaluación anterior.

Los resultados conservarán su procedencia y el dispositivo
registrado en aquella ejecución. El entorno actual no se utilizará
para atribuir los tiempos anteriores.

En modo de nueva evaluación, esta etapa se omitirá.

In [30]:
# ============================================================
# 4.4 Verificación de las evidencias recuperadas
# ============================================================

import json
import hashlib
from pathlib import PurePosixPath
from zipfile import ZipFile

print("=" * 60)
print("🔎 Comprobando la correspondencia de las evidencias...")
print("=" * 60)

evidencias_verificadas = False

if modo_evaluacion == "nueva_evaluacion":
    print("\nNo se utilizarán resultados anteriores.")
    print("La comparación se generará mediante una nueva ejecución.")

elif modo_evaluacion == "recuperacion_evidencias":
    if ruta_zip_evidencias is None:
        raise ValueError("Primero ejecuta 4.3 y carga el respaldo.")

    with ZipFile(ruta_zip_evidencias, "r") as paquete:
        manifest_paquete = json.loads(
            paquete.read(
                "paquete_final/manifest_paquete.json"
            ).decode("utf-8")
        )

        # Verificamos todos los archivos registrados del paquete final
        for nombre, detalle in manifest_paquete["archivos"].items():
            ruta_relativa = PurePosixPath(nombre)

            if ruta_relativa.is_absolute() or ".." in ruta_relativa.parts:
                raise ValueError("El manifiesto contiene una ruta inválida.")

            contenido = paquete.read("paquete_final/" + nombre)
            huella = hashlib.sha256(contenido).hexdigest()

            if huella != detalle["sha256"]:
                raise ValueError(
                    f"No coincide la huella de {nombre}."
                )

        # Comparamos los seis insumos con el paquete ya cargado
        for nombre, detalle in manifest_insumos["archivos"].items():
            contenido = paquete.read("paquete_final/dataset/" + nombre)

            if hashlib.sha256(contenido).hexdigest() != detalle["sha256"]:
                raise ValueError(
                    f"Las evidencias corresponden a otro insumo: {nombre}."
                )

        registro_evaluacion_anterior = json.loads(
            paquete.read(
                "paquete_final/resultados_prueba/registro_evaluacion.json"
            ).decode("utf-8")
        )

        if (
            registro_evaluacion_anterior["version_dataset"]
            != manifest_insumos["version_dataset_utc"]
        ):
            raise ValueError("La versión del dataset no coincide.")

        if (
            registro_evaluacion_anterior["configuracion_prueba"]
            != configuracion_prueba
        ):
            raise ValueError("La configuración de evaluación no coincide.")

        modelos_anteriores = {
            nombre: detalle["identificador"]
            for nombre, detalle
            in registro_evaluacion_anterior["modelos"].items()
        }

        if modelos_anteriores != modelos_comparacion:
            raise ValueError("Los modelos de la evaluación no coinciden.")

        if registro_evaluacion_anterior["consultas"] != len(df_prueba):
            raise ValueError("El número de consultas no coincide.")

        # Comprobamos el archivo realmente utilizado para evaluar
        prueba_anterior = paquete.read(
            "paquete_final/resultados_prueba/dataset_prueba_utilizado.csv"
        )

        huella_prueba = hashlib.sha256(prueba_anterior).hexdigest()

        if huella_prueba != (
            manifest_insumos["archivos"]["dataset_prueba.csv"]["sha256"]
        ):
            raise ValueError(
                "El dataset utilizado para evaluar difiere del actual."
            )

    # Registramos la procedencia sin confundir los dos entornos
    procedencia_resultados = {
        "origen": "evaluacion_previa_primer_colab",
        "ejecutada_en_esta_sesion": False,
        "version_evaluacion": registro_evaluacion_anterior["version_utc"],
        "version_dataset": registro_evaluacion_anterior["version_dataset"],
        "dispositivo_evaluacion": registro_evaluacion_anterior["dispositivo"],
        "archivo_respaldo": ruta_zip_evidencias.name
    }

    evidencias_verificadas = True

    print("\n✅ Huellas del paquete final: correctas.")
    print("✅ Los seis insumos coinciden con los actuales.")
    print("✅ Versión del dataset y 240 consultas: coinciden.")
    print("✅ Modelos y configuración: coinciden.")

    print("\n📋 Procedencia de los resultados:")
    print("Origen: evaluación previa del primer Colab.")
    print(
        "Versión de la evaluación:",
        procedencia_resultados["version_evaluacion"]
    )
    print(
        "GPU utilizada en aquella evaluación:",
        procedencia_resultados["dispositivo_evaluacion"]
    )
    print("Los modelos no se han ejecutado en esta sesión.")
    print("Los tiempos recuperados pertenecen a aquella ejecución.")

else:
    raise ValueError("Modo de evaluación desconocido.")

print("\nSiguiente paso: preparar la evaluación o cargar sus resultados.")

print("=" * 60)
print("✅ Comprobación de la fuente de resultados completada.")
print("=" * 60)

🔎 Comprobando la correspondencia de las evidencias...

✅ Huellas del paquete final: correctas.
✅ Los seis insumos coinciden con los actuales.
✅ Versión del dataset y 240 consultas: coinciden.
✅ Modelos y configuración: coinciden.

📋 Procedencia de los resultados:
Origen: evaluación previa del primer Colab.
Versión de la evaluación: 20261010T200128828138Z
GPU utilizada en aquella evaluación: Tesla T4
Los modelos no se han ejecutado en esta sesión.
Los tiempos recuperados pertenecen a aquella ejecución.

Siguiente paso: preparar la evaluación o cargar sus resultados.
✅ Comprobación de la fuente de resultados completada.


## **📊 5. Evaluación de modelos y recuperación de resultados**

Esta sección utilizará el modo establecido en la preparación:

- Con GPU, se ejecutarán los tres modelos.
- Sin GPU, se cargarán las evidencias verificadas del primer Colab.

Ambos caminos utilizarán la misma estructura de resultados para
generar las tablas y el análisis posterior.

### **🧠 5.1 Preparación de los resultados**

En modo de recuperación se cargarán, para cada modelo,
las predicciones, métricas por categoría, errores, matrices
de confusión y tiempos registrados.

En modo de nueva evaluación se prepararán los contenedores
que recibirán los resultados de la inferencia.

**Procedencia:** Las salidas indicarán si los modelos se ejecutaron
en esta sesión o si se recuperaron resultados anteriores.

In [31]:
# ============================================================
# 5.1 Preparación de los resultados
# ============================================================

import io
import pandas as pd
from zipfile import ZipFile

print("=" * 60)
print("📊 Preparando los resultados de la comparación...")
print("=" * 60)

resultados_modelos = {}
df_comparacion_prueba = pd.DataFrame()
resultados_disponibles = False

if modo_evaluacion == "recuperacion_evidencias":
    if not evidencias_verificadas:
        raise ValueError(
            "Primero ejecuta 4.4 para verificar las evidencias."
        )

    with ZipFile(ruta_zip_evidencias, "r") as paquete:

        # Leemos los CSV directamente desde el respaldo
        def leer_csv_evidencia(nombre, **opciones):
            contenido = paquete.read(
                "paquete_final/resultados_prueba/" + nombre
            )
            return pd.read_csv(
                io.BytesIO(contenido),
                encoding="utf-8-sig",
                **opciones
            )

        df_comparacion_prueba = leer_csv_evidencia(
            "comparacion_prueba.csv"
        )

        for nombre in modelos_comparacion:
            registro_modelo = (
                registro_evaluacion_anterior["modelos"][nombre]
            )
            prefijo = registro_modelo["prefijo_archivos"]

            predicciones = leer_csv_evidencia(
                f"{prefijo}_predicciones.csv",
                keep_default_na=False
            )

            if (
                len(predicciones) != len(df_prueba)
                or not predicciones["id_consulta"].is_unique
                or set(predicciones["id_consulta"])
                != set(df_prueba["id_consulta"])
            ):
                raise ValueError(
                    f"Las predicciones de {nombre} no corresponden "
                    "a las 240 consultas de prueba."
                )

            resultados_modelos[nombre] = {
                "identificador": registro_modelo["identificador"],
                "predicciones": predicciones,
                "metricas_categoria": leer_csv_evidencia(
                    f"{prefijo}_metricas_categoria.csv"
                ),
                "errores": leer_csv_evidencia(
                    f"{prefijo}_errores.csv",
                    keep_default_na=False
                ),
                "matriz_confusion": leer_csv_evidencia(
                    f"{prefijo}_matriz_confusion.csv",
                    index_col=0
                ),
                "metricas": registro_modelo["metricas"],
                "tiempos": registro_modelo["tiempos"],
                "tiempo_carga": registro_modelo["tiempo_carga"],
                "etiquetas_estables": registro_modelo["etiquetas_estables"]
            }

    if (
        len(df_comparacion_prueba) != len(modelos_comparacion)
        or not df_comparacion_prueba["Modelo"].is_unique
        or set(df_comparacion_prueba["Modelo"])
        != set(modelos_comparacion)
    ):
        raise ValueError(
            "La tabla comparativa no contiene los tres modelos esperados."
        )

    resultados_disponibles = True

    print("\nOrigen: resultados recuperados del primer Colab.")
    print(
        "GPU de la evaluación:",
        procedencia_resultados["dispositivo_evaluacion"]
    )
    print("Modelos recuperados:", len(resultados_modelos))
    print("Predicciones por modelo:", len(df_prueba))
    print("Los modelos no se ejecutaron en esta sesión.")

    print("\n📋 Evidencias cargadas por modelo:")
    for nombre in resultados_modelos:
        print("  •", nombre)

    print("\nPara cada modelo se recuperaron:")
    print("  • Predicciones y errores.")
    print("  • Métricas generales y por categoría.")
    print("  • Matriz de confusión.")
    print("  • Tiempos de inferencia y carga.")
    print("  • Estabilidad de etiquetas entre repeticiones.")

elif modo_evaluacion == "nueva_evaluacion":
    print("\nOrigen previsto: nueva evaluación en esta sesión.")
    print("Contenedores de resultados preparados.")
    print("Todavía no se han ejecutado los modelos.")
    print("Los resultados se generarán en el siguiente paso.")

else:
    raise ValueError("Modo de evaluación desconocido.")

print("\nVariables para las siguientes etapas:")
print("  • resultados_modelos: evidencias de cada modelo.")
print("  • df_comparacion_prueba: tabla comparativa.")
print("  • resultados_disponibles: indica si hay resultados cargados.")

print("=" * 60)
print("✅ Preparación de resultados completada.")
print("=" * 60)

📊 Preparando los resultados de la comparación...

Origen: resultados recuperados del primer Colab.
GPU de la evaluación: Tesla T4
Modelos recuperados: 3
Predicciones por modelo: 240
Los modelos no se ejecutaron en esta sesión.

📋 Evidencias cargadas por modelo:
  • MiniLMv2 multilingüe
  • mDeBERTa-v3-base
  • XLM-RoBERTa-large

Para cada modelo se recuperaron:
  • Predicciones y errores.
  • Métricas generales y por categoría.
  • Matriz de confusión.
  • Tiempos de inferencia y carga.
  • Estabilidad de etiquetas entre repeticiones.

Variables para las siguientes etapas:
  • resultados_modelos: evidencias de cada modelo.
  • df_comparacion_prueba: tabla comparativa.
  • resultados_disponibles: indica si hay resultados cargados.
✅ Preparación de resultados completada.


### **🧠 5.2 Ejecución condicional de los modelos**

En modo de nueva evaluación se cargarán los modelos uno por uno,
con precisión float32 y la configuración fijada.

Se realizará calentamiento con dos consultas de desarrollo y
tres repeticiones sobre las 240 consultas de prueba. El tiempo
de carga se medirá por separado.

En modo de recuperación se conservarán los resultados verificados
del primer Colab, sin descargar ni ejecutar los modelos.

**Estado:** Al completar esta etapa estarán disponibles los
resultados para las tablas y el análisis, con su procedencia.

In [32]:
# ============================================================
# 5.2 Ejecución condicional de los modelos
# ============================================================

import gc
import time
import numpy as np
import pandas as pd
import torch
from datetime import datetime, timezone
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix
)

print("=" * 60)
print("🔎 Preparando la evaluación según el modo seleccionado...")
print("=" * 60)

if modo_evaluacion == "recuperacion_evidencias":
    if not evidencias_verificadas or not resultados_disponibles:
        raise ValueError(
            "Primero completa 4.4 y 5.1 para recuperar las evidencias."
        )

    print("\nSe conserva la evaluación previa del primer Colab.")
    print("Modelos recuperados:", len(resultados_modelos))
    print("GPU utilizada en aquella ejecución:",
          procedencia_resultados["dispositivo_evaluacion"])
    print("No se descargarán modelos ni se ejecutará inferencia.")
    print("Los tiempos pertenecen a la ejecución anterior.")

elif modo_evaluacion == "nueva_evaluacion":
    if not torch.cuda.is_available():
        raise RuntimeError(
            "La GPU ya no está disponible. Vuelve a ejecutar 2.2. "
            "Si selecciona recuperación, completa 4.3, 4.4 y 5.1."
        )

    # Importamos el pipeline solo cuando vamos a ejecutar modelos
    from transformers import pipeline

    resultados_disponibles = False
    resultados_modelos = {}
    filas_metricas = []
    consultas = df_prueba["consulta_es"].tolist()
    referencias = df_prueba["categoria_referencia"].tolist()

    version_evaluacion = datetime.now(timezone.utc).strftime(
        "%Y%m%dT%H%M%S%fZ"
    )

    for numero, (nombre, identificador) in enumerate(
        modelos_comparacion.items(), start=1
    ):
        print("\n" + "=" * 60)
        print(f"Modelo {numero}/3: {nombre}")
        print("=" * 60)

        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        inicio_carga = time.perf_counter()
        clasificador = None

        try:
            clasificador = pipeline(
                "zero-shot-classification",
                model=identificador,
                device=0,
                dtype=torch.float32
            )
            torch.cuda.synchronize()
            tiempo_carga = time.perf_counter() - inicio_carga

            parametros = {
                "candidate_labels": etiquetas_candidatas,
                "hypothesis_template": configuracion_prueba["plantilla"],
                "multi_label": configuracion_prueba["multi_label"],
                "batch_size": configuracion_prueba["batch_size"]
            }

            # Calentamiento fuera de las mediciones
            with torch.inference_mode():
                clasificador(
                    df_desarrollo["consulta_es"].tolist()[:2],
                    **parametros
                )
            torch.cuda.synchronize()

            tiempos = []
            etiquetas_repeticiones = []
            salida_primera = None

            for repeticion in range(configuracion_prueba["repeticiones"]):
                torch.cuda.synchronize()
                inicio = time.perf_counter()

                with torch.inference_mode():
                    salida = clasificador(consultas, **parametros)

                torch.cuda.synchronize()
                tiempos.append(time.perf_counter() - inicio)

                etiquetas_repeticiones.append([
                    descripcion_a_categoria[r["labels"][0]]
                    for r in salida
                ])

                if repeticion == 0:
                    salida_primera = salida

                print(
                    f"Repetición {repeticion + 1}: "
                    f"{tiempos[-1]:.2f} segundos"
                )

            predichas = etiquetas_repeticiones[0]
            estables = all(
                etiquetas == predichas
                for etiquetas in etiquetas_repeticiones[1:]
            )

            precision, recall, f1, soporte = (
                precision_recall_fscore_support(
                    referencias,
                    predichas,
                    labels=categorias,
                    average=None,
                    zero_division=0
                )
            )

            metricas = {
                "Modelo": nombre,
                "Exactitud": float(accuracy_score(referencias, predichas)),
                "Precisión macro": float(np.mean(precision)),
                "Exhaustividad macro": float(np.mean(recall)),
                "F1 macro": float(np.mean(f1)),
                "Tiempo medio total (s)": float(np.mean(tiempos)),
                "Desviación de tiempo (s)": float(
                    np.std(tiempos, ddof=1)
                ),
                "Tiempo medio por consulta (s)": float(
                    np.mean(tiempos) / len(consultas)
                )
            }

            predicciones = df_prueba.copy()
            predicciones["categoria_predicha"] = predichas
            predicciones["puntuacion"] = [
                float(r["scores"][0]) for r in salida_primera
            ]
            predicciones["acierto"] = (
                predicciones["categoria_referencia"]
                .eq(predicciones["categoria_predicha"])
            )

            matriz = pd.DataFrame(
                confusion_matrix(
                    referencias, predichas, labels=categorias
                ),
                index=categorias,
                columns=categorias
            )
            matriz.index.name = "Categoría de referencia"
            matriz.columns.name = "Categoría predicha"

            resultados_modelos[nombre] = {
                "identificador": identificador,
                "predicciones": predicciones,
                "errores": predicciones.loc[
                    ~predicciones["acierto"]
                ].copy(),
                "metricas_categoria": pd.DataFrame({
                    "Categoría": categorias,
                    "Precisión": precision,
                    "Exhaustividad": recall,
                    "F1": f1,
                    "Consultas": soporte
                }),
                "matriz_confusion": matriz,
                "metricas": metricas,
                "tiempos": [float(t) for t in tiempos],
                "tiempo_carga": float(tiempo_carga),
                "etiquetas_estables": bool(estables)
            }

            filas_metricas.append(metricas)
            display(pd.DataFrame([metricas]).round(4))
            print("Aciertos:", int(predicciones["acierto"].sum()), "de 240")
            print("Etiquetas estables:", estables)

        finally:
            # Liberamos el modelo incluso si ocurre un error
            del clasificador
            gc.collect()
            torch.cuda.empty_cache()

    df_comparacion_prueba = (
        pd.DataFrame(filas_metricas)
        .sort_values("F1 macro", ascending=False)
        .reset_index(drop=True)
    )

    procedencia_resultados = {
        "origen": "nueva_evaluacion_segundo_colab",
        "ejecutada_en_esta_sesion": True,
        "version_evaluacion": version_evaluacion,
        "version_dataset": manifest_insumos["version_dataset_utc"],
        "dispositivo_evaluacion": torch.cuda.get_device_name(0),
        "archivo_respaldo": None
    }
    resultados_disponibles = True

    print("\nLos tres modelos se ejecutaron en esta sesión.")

else:
    raise ValueError("Modo de evaluación desconocido.")

print("\nSiguiente paso: comprobar y presentar las métricas.")
print("Los archivos se exportarán en la etapa de respaldo.")

print("=" * 60)
print("✅ Resultados disponibles para la comparación.")
print("=" * 60)

🔎 Preparando la evaluación según el modo seleccionado...

Se conserva la evaluación previa del primer Colab.
Modelos recuperados: 3
GPU utilizada en aquella ejecución: Tesla T4
No se descargarán modelos ni se ejecutará inferencia.
Los tiempos pertenecen a la ejecución anterior.

Siguiente paso: comprobar y presentar las métricas.
Los archivos se exportarán en la etapa de respaldo.
✅ Resultados disponibles para la comparación.


### **🧠 5.3 Comprobación y presentación de métricas generales**

Se recalcularán exactitud, precisión macro, exhaustividad macro
y F1 macro a partir de las predicciones disponibles.

Se comprobará que las consultas y categorías de referencia
coincidan con el dataset de prueba y que las métricas calculadas
correspondan a las registradas.

La tabla comparativa incluirá aciertos y tiempos de ejecución.
Los tiempos conservarán su procedencia: no se medirán nuevamente
en el modo de recuperación.

**Nota:** Recalcular métricas no equivale a ejecutar los modelos.

In [33]:
# ============================================================
# 5.3 Comprobación y presentación de métricas generales
# ============================================================

import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support
)

print("=" * 60)
print("📊 Comprobando las métricas desde las predicciones...")
print("=" * 60)

if not resultados_disponibles:
    raise ValueError("Primero completa 5.1 y 5.2.")

# Establecemos el mismo orden de consultas para todos los modelos
referencia = (
    df_prueba.set_index("id_consulta")
    [["consulta_es", "categoria_referencia"]]
)

filas_comprobadas = []

for nombre in modelos_comparacion:
    resultado = resultados_modelos[nombre]
    predicciones = resultado["predicciones"]

    campos = {
        "id_consulta",
        "consulta_es",
        "categoria_referencia",
        "categoria_predicha"
    }

    if not campos.issubset(predicciones.columns):
        raise ValueError(f"Faltan campos en las predicciones de {nombre}.")

    if (
        not predicciones["id_consulta"].is_unique
        or set(predicciones["id_consulta"]) != set(referencia.index)
    ):
        raise ValueError(f"Los identificadores de {nombre} no coinciden.")

    ordenadas = (
        predicciones.set_index("id_consulta")
        .loc[referencia.index]
    )

    if not ordenadas[
        ["consulta_es", "categoria_referencia"]
    ].equals(referencia):
        raise ValueError(
            f"Las consultas o referencias de {nombre} no coinciden."
        )

    if not ordenadas["categoria_predicha"].isin(categorias).all():
        raise ValueError(f"Hay categorías predichas inválidas en {nombre}.")

    reales = referencia["categoria_referencia"].tolist()
    predichas = ordenadas["categoria_predicha"].tolist()

    precision, recall, f1, _ = precision_recall_fscore_support(
        reales,
        predichas,
        labels=categorias,
        average="macro",
        zero_division=0
    )

    metricas_calculadas = {
        "Exactitud": float(accuracy_score(reales, predichas)),
        "Precisión macro": float(precision),
        "Exhaustividad macro": float(recall),
        "F1 macro": float(f1)
    }

    # Comparamos los valores calculados con el registro original
    for campo, valor in metricas_calculadas.items():
        if not np.isclose(
            valor,
            resultado["metricas"][campo],
            rtol=0,
            atol=1e-10
        ):
            raise ValueError(
                f"La métrica {campo} de {nombre} no coincide."
            )

    tiempos = np.asarray(resultado["tiempos"], dtype=float)

    if (
        len(tiempos) != configuracion_prueba["repeticiones"]
        or not np.isfinite(tiempos).all()
        or (tiempos <= 0).any()
    ):
        raise ValueError(f"Los tiempos registrados de {nombre} son inválidos.")

    tiempos_calculados = {
        "Tiempo medio total (s)": float(tiempos.mean()),
        "Desviación de tiempo (s)": float(tiempos.std(ddof=1)),
        "Tiempo medio por consulta (s)": float(
            tiempos.mean() / len(referencia)
        )
    }

    for campo, valor in tiempos_calculados.items():
        if not np.isclose(
            valor,
            resultado["metricas"][campo],
            rtol=0,
            atol=1e-10
        ):
            raise ValueError(
                f"El resumen de tiempo {campo} de {nombre} no coincide."
            )

    filas_comprobadas.append({
        "Modelo": nombre,
        **metricas_calculadas,
        "Aciertos": int(
            ordenadas["categoria_referencia"]
            .eq(ordenadas["categoria_predicha"]).sum()
        ),
        **tiempos_calculados,
        "Tiempo de carga (s)": float(resultado["tiempo_carga"]),
        "Etiquetas estables": bool(resultado["etiquetas_estables"])
    })

# Esta tabla común servirá para análisis y exportación
df_comparacion_prueba = (
    pd.DataFrame(filas_comprobadas)
    .sort_values("F1 macro", ascending=False)
    .reset_index(drop=True)
)

print(
    "\nOrigen:",
    "Nueva evaluación en esta sesión."
    if procedencia_resultados["ejecutada_en_esta_sesion"]
    else "Evaluación previa del primer Colab."
)
print(
    "Dispositivo de la evaluación:",
    procedencia_resultados["dispositivo_evaluacion"]
)

print("\n📋 Comparación de calidad de clasificación:")
display(
    df_comparacion_prueba[
        [
            "Modelo",
            "Exactitud",
            "Precisión macro",
            "Exhaustividad macro",
            "F1 macro",
            "Aciertos"
        ]
    ].round(4)
)

print("\n⏱️ Comparación de tiempos y estabilidad:")
display(
    df_comparacion_prueba[
        [
            "Modelo",
            "Tiempo medio total (s)",
            "Desviación de tiempo (s)",
            "Tiempo medio por consulta (s)",
            "Tiempo de carga (s)",
            "Etiquetas estables"
        ]
    ].round(4)
)

print("\nConsultas evaluadas por modelo:", len(df_prueba))
print("Métricas comprobadas desde las predicciones: correctas.")
print("Resúmenes de tiempo comprobados desde las repeticiones: correctos.")
print("El tiempo por consulta es un promedio del procesamiento completo.")
print("Siguiente paso: resultados por categoría y análisis de errores.")

print("=" * 60)
print("✅ Métricas verificadas y comparación general presentada.")
print("=" * 60)

📊 Comprobando las métricas desde las predicciones...

Origen: Evaluación previa del primer Colab.
Dispositivo de la evaluación: Tesla T4

📋 Comparación de calidad de clasificación:


,Modelo,Exactitud,Precisión macro,Exhaustividad macro,F1 macro,Aciertos
0,XLM-RoBERTa-large,0.6583,0.7460,0.6583,0.6492,158
1,MiniLMv2 multilingüe,0.4417,0.4116,0.4417,0.4002,106
2,mDeBERTa-v3-base,0.3125,0.5596,0.3125,0.2893,75



⏱️ Comparación de tiempos y estabilidad:


,Modelo,Tiempo medio total (s),Desviación de tiempo (s),Tiempo medio por consulta (s),Tiempo de carga (s),Etiquetas estables
0,XLM-RoBERTa-large,26.6031,0.0942,0.1108,5.4520,True
1,MiniLMv2 multilingüe,18.5258,5.9288,0.0772,8.4253,True
2,mDeBERTa-v3-base,34.0984,0.2110,0.1421,4.1721,True



Consultas evaluadas por modelo: 240
Métricas comprobadas desde las predicciones: correctas.
Resúmenes de tiempo comprobados desde las repeticiones: correctos.
El tiempo por consulta es un promedio del procesamiento completo.
Siguiente paso: resultados por categoría y análisis de errores.
✅ Métricas verificadas y comparación general presentada.


### **🧠 5.4 Resultados por categoría y análisis de errores**

Se calcularán las métricas por categoría y las matrices de
confusión a partir de las predicciones verificadas.

Se mostrarán los aciertos por categoría y las confusiones
más frecuentes de cada modelo.

Las matrices utilizarán filas para las categorías de referencia
y columnas para las categorías predichas.

**Alcance:** Este análisis permite identificar debilidades que
el promedio general puede ocultar. No se modificarán las
descripciones ni los modelos a partir de los errores de prueba.

In [34]:
# ============================================================
# 5.4 Resultados por categoría y análisis de errores
# ============================================================

import numpy as np
import pandas as pd
from sklearn.metrics import (
    precision_recall_fscore_support,
    confusion_matrix
)

print("=" * 60)
print("🔎 Analizando categorías y errores de clasificación...")
print("=" * 60)

if not resultados_disponibles:
    raise ValueError("Primero completa la preparación de resultados.")

tablas_categoria = []
tablas_confusiones = []
resumen_aciertos_categoria = pd.DataFrame(index=categorias)
resumen_aciertos_categoria.index.name = "Categoría"

for nombre in modelos_comparacion:
    resultado = resultados_modelos[nombre]

    # Alineamos las predicciones con el orden del dataset de prueba
    predicciones = (
        resultado["predicciones"]
        .set_index("id_consulta")
        .loc[df_prueba["id_consulta"]]
        .reset_index()
        .copy()
    )

    reales = predicciones["categoria_referencia"]
    predichas = predicciones["categoria_predicha"]

    precision, recall, f1, soporte = (
        precision_recall_fscore_support(
            reales,
            predichas,
            labels=categorias,
            average=None,
            zero_division=0
        )
    )

    matriz = pd.DataFrame(
        confusion_matrix(reales, predichas, labels=categorias),
        index=categorias,
        columns=categorias
    )
    matriz.index.name = "Categoría de referencia"
    matriz.columns.name = "Categoría predicha"

    tabla_categoria = pd.DataFrame({
        "Categoría": categorias,
        "Precisión": precision,
        "Exhaustividad": recall,
        "F1": f1,
        "Consultas": soporte,
        "Aciertos": np.diag(matriz.to_numpy())
    })

    # Comprobamos las métricas por categoría registradas
    tabla_registrada = (
        resultado["metricas_categoria"]
        .set_index("Categoría")
        .loc[categorias]
    )

    for campo in ["Precisión", "Exhaustividad", "F1", "Consultas"]:
        if not np.allclose(
            tabla_categoria[campo].to_numpy(),
            tabla_registrada[campo].to_numpy(),
            rtol=0,
            atol=1e-10
        ):
            raise ValueError(
                f"Las métricas por categoría de {nombre} no coinciden."
            )

    matriz_registrada = (
        resultado["matriz_confusion"]
        .loc[categorias, categorias]
    )

    if not np.array_equal(
        matriz.to_numpy(),
        matriz_registrada.to_numpy()
    ):
        raise ValueError(
            f"La matriz de confusión de {nombre} no coincide."
        )

    # Recalculamos los errores desde las etiquetas verificadas
    predicciones["acierto"] = reales.eq(predichas)
    errores = predicciones.loc[
        ~predicciones["acierto"]
    ].copy()

    errores_registrados = resultado["errores"]

    if (
        len(errores_registrados) != len(errores)
        or set(errores_registrados["id_consulta"])
        != set(errores["id_consulta"])
    ):
        raise ValueError(
            f"Los identificadores de errores de {nombre} no coinciden."
        )

    confusiones = (
        errores.groupby(
            ["categoria_referencia", "categoria_predicha"]
        )
        .size()
        .reset_index(name="Consultas")
        .sort_values(
            ["Consultas", "categoria_referencia", "categoria_predicha"],
            ascending=[False, True, True]
        )
        .reset_index(drop=True)
    )

    # Conservamos las tablas calculadas para el análisis y exportación
    resultado["metricas_categoria"] = tabla_categoria
    resultado["matriz_confusion"] = matriz
    resultado["predicciones"] = predicciones
    resultado["errores"] = errores
    resultado["confusiones_frecuentes"] = confusiones

    resumen_aciertos_categoria[nombre] = (
        tabla_categoria["Aciertos"].to_numpy()
    )

    tablas_categoria.append(
        tabla_categoria.assign(Modelo=nombre)
    )
    tablas_confusiones.append(
        confusiones.assign(Modelo=nombre)
    )

    print("\n" + "=" * 60)
    print("Modelo:", nombre)
    print("=" * 60)

    print("\n📋 Resultados por categoría:")
    display(tabla_categoria.round(4))

    print("\n📊 Matriz de confusión:")
    display(matriz)

    print("\n🔎 Cinco confusiones más frecuentes:")
    display(confusiones.head(5))

    print("Consultas clasificadas incorrectamente:", len(errores))

df_metricas_por_categoria = pd.concat(
    tablas_categoria, ignore_index=True
)

df_confusiones_frecuentes = pd.concat(
    tablas_confusiones, ignore_index=True
)

print("\n📋 Aciertos por categoría —40 consultas por categoría—:")
display(resumen_aciertos_categoria.reset_index())

print("\nOrigen de la evaluación:", procedencia_resultados["origen"])
print("Métricas por categoría y matrices: comprobadas.")
print("Los errores completos están disponibles en resultados_modelos.")
print("Siguiente paso: interpretación y selección del modelo.")

print("=" * 60)
print("✅ Resultados por categoría y errores analizados.")
print("=" * 60)

🔎 Analizando categorías y errores de clasificación...

Modelo: MiniLMv2 multilingüe

📋 Resultados por categoría:


,Categoría,Precisión,Exhaustividad,F1,Consultas,Aciertos
0,LIFE_INSURANCE,0.1111,0.025,0.0408,40,1
1,COVERAGE,0.4074,0.275,0.3284,40,11
2,QUOTE,0.6538,0.425,0.5152,40,17
3,PAYMENT,0.1190,0.125,0.1220,40,5
4,ENROLLMENT,0.3864,0.850,0.5312,40,34
5,CONTACT,0.7917,0.950,0.8636,40,38



📊 Matriz de confusión:


Categoría predicha,LIFE_INSURANCE,COVERAGE,QUOTE,PAYMENT,ENROLLMENT,CONTACT
Categoría de referencia,,,,,,
LIFE_INSURANCE,1,7,1,28,0,3
COVERAGE,3,11,6,8,7,5
QUOTE,4,5,17,1,13,0
PAYMENT,0,0,1,5,32,2
ENROLLMENT,1,4,1,0,34,0
CONTACT,0,0,0,0,2,38



🔎 Cinco confusiones más frecuentes:


...


🔎 Cinco confusiones más frecuentes:


,categoria_referencia,categoria_predicha,Consultas
0,QUOTE,LIFE_INSURANCE,17
1,COVERAGE,LIFE_INSURANCE,16
2,PAYMENT,CONTACT,10
3,PAYMENT,QUOTE,9
4,ENROLLMENT,LIFE_INSURANCE,8


Consultas clasificadas incorrectamente: 82

📋 Aciertos por categoría —40 consultas por categoría—:


,Categoría,MiniLMv2 multilingüe,mDeBERTa-v3-base,XLM-RoBERTa-large
0,LIFE_INSURANCE,1,0,40
1,COVERAGE,11,6,24
2,QUOTE,17,18,22
3,PAYMENT,5,4,17
4,ENROLLMENT,34,10,17
5,CONTACT,38,37,38



Origen de la evaluación: evaluacion_previa_primer_colab
Métricas por categoría y matrices: comprobadas.
Los errores completos están disponibles en resultados_modelos.
Siguiente paso: interpretación y selección del modelo.
✅ Resultados por categoría y errores analizados.


## **📝 6. Interpretación y selección del modelo**

Se interpretarán los resultados considerando el criterio principal
de F1 macro, los tiempos y las diferencias entre categorías.

La recomendación se limita al dataset y la configuración evaluados.
No implica que el modelo esté listo para operar sin supervisión
en un servicio de orientación sobre seguros.

### **🧠 6.1 Selección y balance entre calidad y tiempo**

Se seleccionará el modelo con mayor F1 macro y se comparará
con el modelo de menor tiempo medio de inferencia.

Se registrarán los aciertos, errores y confusiones principales
del modelo seleccionado, conservando la procedencia de la evaluación.

Los resultados servirán para justificar la recomendación del
prototipo y documentar sus limitaciones.

In [35]:
# ============================================================
# 6.1 Selección y balance entre calidad y tiempo
# ============================================================

print("=" * 60)
print("📝 Interpretando los resultados y registrando la selección...")
print("=" * 60)

if not resultados_disponibles or len(df_comparacion_prueba) != 3:
    raise ValueError("Primero completa la comparación de los tres modelos.")

# Aplicamos el criterio establecido antes de evaluar
mejor_f1 = df_comparacion_prueba["F1 macro"].max()
candidatos = df_comparacion_prueba.loc[
    df_comparacion_prueba["F1 macro"].eq(mejor_f1)
]

if len(candidatos) != 1:
    raise ValueError(
        "Hay un empate en F1 macro. Debe documentarse un criterio "
        "adicional antes de registrar una selección única."
    )

seleccionado = candidatos.iloc[0]
modelo_seleccionado = seleccionado["Modelo"]

mas_rapido = df_comparacion_prueba.loc[
    df_comparacion_prueba["Tiempo medio total (s)"].idxmin()
]

resultado_seleccionado = resultados_modelos[modelo_seleccionado]
numero_errores = len(resultado_seleccionado["errores"])

# Comparamos las mejores métricas por categoría
mejores_por_categoria = df_metricas_por_categoria.loc[
    df_metricas_por_categoria.groupby("Categoría")["F1"].transform("max")
    .eq(df_metricas_por_categoria["F1"]),
    ["Categoría", "Modelo", "F1"]
].copy()

mejores_por_categoria["Categoría"] = pd.Categorical(
    mejores_por_categoria["Categoría"],
    categories=categorias,
    ordered=True
)

mejores_por_categoria = (
    mejores_por_categoria
    .sort_values(["Categoría", "Modelo"])
    .reset_index(drop=True)
)

# Conservamos una decisión reutilizable para el respaldo
decision_final = {
    "modelo_seleccionado": modelo_seleccionado,
    "identificador": modelos_comparacion[modelo_seleccionado],
    "criterio_principal": "mayor F1 macro sobre prueba",
    "f1_macro": float(seleccionado["F1 macro"]),
    "exactitud": float(seleccionado["Exactitud"]),
    "aciertos": int(seleccionado["Aciertos"]),
    "errores": int(numero_errores),
    "consultas_prueba": len(df_prueba),
    "modelo_mas_rapido": mas_rapido["Modelo"],
    "procedencia": dict(procedencia_resultados),
    "alcance": "Recomendación para el prototipo con la configuración evaluada.",
    "estado_operativo": "Requiere validación adicional antes de producción."
}

print("\n📋 Procedencia:")
print(
    "Nueva evaluación en esta sesión."
    if procedencia_resultados["ejecutada_en_esta_sesion"]
    else "Evaluación previa del primer Colab, recuperada y comprobada."
)
print(
    "Dispositivo utilizado:",
    procedencia_resultados["dispositivo_evaluacion"]
)

print("\n🏆 Modelo seleccionado:", modelo_seleccionado)
print("Criterio: mayor F1 macro sobre prueba.")
print(f"F1 macro: {seleccionado['F1 macro']:.4f}")
print(f"Exactitud: {seleccionado['Exactitud']:.2%}")
print("Aciertos:", int(seleccionado["Aciertos"]), "de", len(df_prueba))
print("Errores:", numero_errores)

print("\n⏱️ Balance entre calidad y tiempo:")
print("Modelo con menor tiempo medio:", mas_rapido["Modelo"])
print(
    f"Tiempo medio del más rápido: "
    f"{mas_rapido['Tiempo medio total (s)']:.4f} segundos"
)
print(
    f"Tiempo medio del seleccionado: "
    f"{seleccionado['Tiempo medio total (s)']:.4f} segundos"
)

if modelo_seleccionado != mas_rapido["Modelo"]:
    diferencia_f1 = (
        seleccionado["F1 macro"] - mas_rapido["F1 macro"]
    )
    diferencia_tiempo = (
        seleccionado["Tiempo medio total (s)"]
        - mas_rapido["Tiempo medio total (s)"]
    )
    print(
        f"Frente al más rápido, el seleccionado gana "
        f"{diferencia_f1:.4f} en F1 macro y requiere "
        f"{diferencia_tiempo:.4f} segundos adicionales "
        "por ejecución del conjunto completo."
    )

print("\n📋 Mayor F1 por categoría —se incluyen empates—:")
display(mejores_por_categoria.round(4))

print("\n🔎 Principales confusiones del modelo seleccionado:")
display(
    resultado_seleccionado["confusiones_frecuentes"].head(5)
)

print("\nLimitaciones de la recomendación:")
print("  • Se evaluaron 240 consultas traducidas y revisadas.")
print("  • Pueden persistir consultas semánticamente similares.")
print("  • El mejor resultado global no garantiza el mejor resultado en cada categoría.")
print("  • Tres repeticiones sobre los mismos datos no prueban generalización.")
print("  • Los puntajes de clasificación no son probabilidades calibradas.")
print("  • No se midieron costos monetarios ni consumo máximo de memoria.")
print("  • No se ha validado un umbral de aceptación para uso operativo.")

print("\nDecisión conservada en: decision_final")
print("Siguiente paso: documentar ajustes, riesgos y uso posterior.")

print("=" * 60)
print("✅ Selección registrada y balance entre calidad y tiempo documentado.")
print("=" * 60)

📝 Interpretando los resultados y registrando la selección...

📋 Procedencia:
Evaluación previa del primer Colab, recuperada y comprobada.
Dispositivo utilizado: Tesla T4

🏆 Modelo seleccionado: XLM-RoBERTa-large
Criterio: mayor F1 macro sobre prueba.
F1 macro: 0.6492
Exactitud: 65.83%
Aciertos: 158 de 240
Errores: 82

⏱️ Balance entre calidad y tiempo:
Modelo con menor tiempo medio: MiniLMv2 multilingüe
Tiempo medio del más rápido: 18.5258 segundos
Tiempo medio del seleccionado: 26.6031 segundos
Frente al más rápido, el seleccionado gana 0.2490 en F1 macro y requiere 8.0773 segundos adicionales por ejecución del conjunto completo.

📋 Mayor F1 por categoría —se incluyen empates—:


,Categoría,Modelo,F1
0,LIFE_INSURANCE,XLM-RoBERTa-large,0.6400
1,COVERAGE,XLM-RoBERTa-large,0.6957
2,QUOTE,mDeBERTa-v3-base,0.6102
3,PAYMENT,XLM-RoBERTa-large,0.5667
4,ENROLLMENT,XLM-RoBERTa-large,0.5965
5,CONTACT,MiniLMv2 multilingüe,0.8636



🔎 Principales confusiones del modelo seleccionado:


,categoria_referencia,categoria_predicha,Consultas
0,QUOTE,LIFE_INSURANCE,17
1,COVERAGE,LIFE_INSURANCE,16
2,PAYMENT,CONTACT,10
3,PAYMENT,QUOTE,9
4,ENROLLMENT,LIFE_INSURANCE,8



Limitaciones de la recomendación:
  • Se evaluaron 240 consultas traducidas y revisadas.
  • Pueden persistir consultas semánticamente similares.
  • El mejor resultado global no garantiza el mejor resultado en cada categoría.
  • Tres repeticiones sobre los mismos datos no prueban generalización.
  • Los puntajes de clasificación no son probabilidades calibradas.
  • No se midieron costos monetarios ni consumo máximo de memoria.
  • No se ha validado un umbral de aceptación para uso operativo.

Decisión conservada en: decision_final
Siguiente paso: documentar ajustes, riesgos y uso posterior.
✅ Selección registrada y balance entre calidad y tiempo documentado.


### **🧠 6.2 Ajustes mínimos, riesgos y condiciones de uso**

#### **Ajustes realizados**

Se utilizaron modelos preentrenados mediante clasificación
zero-shot, sin entrenamiento adicional ni modificación de sus pesos.

La preparación incluyó selección de categorías, traducción al
español, revisión de etiquetas y sustitución de registros con
coincidencias textuales detectadas, conservando la trazabilidad.

En desarrollo se compararon descripciones iniciales y alternativas
de las categorías. Se conservó la configuración inicial común para
evaluar los tres modelos sobre prueba.

Los resultados de prueba no se utilizaron para modificar las
descripciones ni ajustar los modelos.

#### **Interpretación de la recomendación**

El modelo seleccionado en 6.1 obtuvo el mayor F1 macro entre
los tres modelos comparados.

La procedencia indicada en esa sección determina si los resultados
se generaron en esta sesión con GPU o se recuperaron de la
evaluación verificada del primer Colab.

En ambos casos, la recomendación se limita al dataset y la
configuración evaluados. Requiere validación adicional para
aplicarse a consultas reales de usuarios.

Los resultados por categoría muestran que el mejor modelo global
puede ser superado por otro en categorías específicas.

#### **Umbrales y revisión humana**

No se estableció ni validó un umbral numérico para aceptar
automáticamente una clasificación.

Los puntajes del pipeline no se consideran probabilidades
calibradas. En una etapa posterior, cualquier umbral deberá
definirse y validarse con datos de desarrollo o validación
adicionales, considerando los errores y la cobertura de atención.

Antes de un uso operativo se deberán definir criterios para
derivar consultas ambiguas o fuera de alcance a revisión humana.

#### **Riesgos y limitaciones**

- Las consultas proceden de un dataset traducido y revisado;
  su desempeño puede diferir del observado en conversaciones reales.
- La ausencia de coincidencias textuales no garantiza independencia
  semántica entre desarrollo y prueba.
- La evaluación contiene seis categorías y no incluye una categoría
  específica para consultas fuera de alcance.
- Una clasificación incorrecta puede dirigir la consulta a un
  proceso equivocado del asistente.
- La clasificación no confirma coberturas, precios, exclusiones
  ni condiciones contractuales de un seguro.
- Los tiempos corresponden al dispositivo y protocolo registrados.
  No se midieron costos monetarios ni consumo máximo de memoria.
- La estabilidad entre repeticiones no demuestra generalización.

#### **Continuidad del proyecto**

Como trabajo futuro, el PDF de condiciones generales del seguro
de vida podrá evaluarse como fuente documental para responder
consultas mediante recuperación de información y referencias
a páginas.

Antes de incorporarlo se deberá comprobar su versión, producto
y aplicabilidad. El documento no se utilizó como entrada de los
modelos en esta comparación.

**Estado:** Selección experimental documentada. Pendientes:
respaldo final, reporte técnico y organización del repositorio.

## **📦 7. Respaldo final y evidencias para el reporte**

Se exportarán los resultados disponibles y los insumos utilizados,
indicando si la evaluación se ejecutó en esta sesión o se recuperó
del primer Colab.

### **🧠 7.1 Exportación y descarga del paquete final**

El paquete incluirá:

- Datasets, configuración, trazabilidad y auditoría.
- Métricas, predicciones, errores y matrices de confusión.
- Decisión final y procedencia de la evaluación.
- Registro del entorno actual, separado del entorno de evaluación.
- Inventario de archivos con huellas SHA-256.
- Guía de uso para continuar con el reporte.

En modo de recuperación se conservará también el ZIP original
de evidencias, que incluye los experimentos de desarrollo.

**Importante:** El notebook debe descargarse por separado,
con sus salidas, como evidencia del procedimiento y del código.

In [36]:
# ============================================================
# 7.1 Exportación y descarga del paquete final
# ============================================================

import json
import hashlib
import shutil
from pathlib import Path
from datetime import datetime, timezone
from zipfile import ZipFile
from google.colab import files

print("=" * 60)
print("📦 Preparando el respaldo final de la comparación...")
print("=" * 60)

if not resultados_disponibles or len(resultados_modelos) != 3:
    raise ValueError("Primero completa la evaluación o recuperación.")

if "decision_final" not in globals():
    raise ValueError("Primero ejecuta 6.1 para registrar la selección.")

marca_respaldo = datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%S%fZ"
)

carpeta_respaldo = (
    Path("/content/entrega_actividad_4")
    / f"comparacion_final_{marca_respaldo}"
)

carpeta_datos = carpeta_respaldo / "dataset"
carpeta_resultados = carpeta_respaldo / "resultados"
carpeta_registros = carpeta_respaldo / "registros"

for carpeta in [
    carpeta_datos,
    carpeta_resultados,
    carpeta_registros
]:
    carpeta.mkdir(parents=True, exist_ok=False)

def guardar_json(ruta, contenido):
    with ruta.open("w", encoding="utf-8") as archivo:
        json.dump(
            contenido, archivo,
            ensure_ascii=False, indent=2, allow_nan=False
        )

# Copiamos los ocho insumos originales sin modificar sus contenidos
for nombre in archivos_paquete:
    shutil.copy2(
        carpeta_insumos / nombre,
        carpeta_datos / nombre
    )

# Guardamos las tablas comunes a ambos modos
df_comparacion_prueba.to_csv(
    carpeta_resultados / "comparacion_prueba.csv",
    index=False, encoding="utf-8-sig"
)

df_metricas_por_categoria.to_csv(
    carpeta_resultados / "metricas_por_categoria.csv",
    index=False, encoding="utf-8-sig"
)

df_confusiones_frecuentes.to_csv(
    carpeta_resultados / "confusiones_frecuentes.csv",
    index=False, encoding="utf-8-sig"
)

df_prueba.to_csv(
    carpeta_resultados / "dataset_prueba_utilizado.csv",
    index=False, encoding="utf-8-sig"
)

# Conservamos resultados individuales y relación entre modelos y archivos
registro_resultados = {
    "procedencia": dict(procedencia_resultados),
    "configuracion_prueba": configuracion_prueba,
    "consultas": len(df_prueba),
    "modelos": {}
}

for numero, (nombre, resultado) in enumerate(
    resultados_modelos.items(), start=1
):
    prefijo = f"modelo_{numero}"

    for tipo in [
        "predicciones",
        "errores",
        "metricas_categoria",
        "confusiones_frecuentes"
    ]:
        resultado[tipo].to_csv(
            carpeta_resultados / f"{prefijo}_{tipo}.csv",
            index=False, encoding="utf-8-sig"
        )

    resultado["matriz_confusion"].to_csv(
        carpeta_resultados / f"{prefijo}_matriz_confusion.csv",
        encoding="utf-8-sig"
    )

    registro_resultados["modelos"][nombre] = {
        "identificador": modelos_comparacion[nombre],
        "prefijo_archivos": prefijo,
        "metricas": resultado["metricas"],
        "tiempos": resultado["tiempos"],
        "tiempo_carga": resultado["tiempo_carga"],
        "etiquetas_estables": resultado["etiquetas_estables"]
    }

guardar_json(
    carpeta_registros / "registro_resultados.json",
    registro_resultados
)
guardar_json(
    carpeta_registros / "entorno_actual_notebook.json",
    registro_entorno
)
guardar_json(
    carpeta_respaldo / "decision_final.json",
    decision_final
)
guardar_json(
    carpeta_respaldo / "configuracion_modelo_seleccionado.json",
    {
        "modelo": modelo_seleccionado,
        "identificador": modelos_comparacion[modelo_seleccionado],
        "configuracion": configuracion_prueba,
        "procedencia": procedencia_resultados,
        "pesos_incluidos": False
    }
)

# Conservamos el registro histórico y todas las evidencias originales
if modo_evaluacion == "recuperacion_evidencias":
    guardar_json(
        carpeta_registros / "registro_evaluacion_original.json",
        registro_evaluacion_anterior
    )

    carpeta_originales = carpeta_respaldo / "evidencias_originales"
    carpeta_originales.mkdir()

    shutil.copy2(
        ruta_zip_evidencias,
        carpeta_originales / "EVIDENCIAS_PRIMER_COLAB.zip"
    )

    descripcion_origen = (
        "Resultados recuperados de la evaluación del primer Colab. "
        "Los modelos no se ejecutaron en esta sesión."
    )
else:
    guardar_json(
        carpeta_registros / "entorno_evaluacion.json",
        {
            **registro_entorno,
            "gpu": procedencia_resultados["dispositivo_evaluacion"]
        }
    )
    descripcion_origen = (
        "Resultados generados mediante una nueva evaluación "
        "de los tres modelos en esta sesión."
    )

# Documentamos qué utilizar para continuar
guia = f"""PAQUETE FINAL DE COMPARACIÓN — ACTIVIDAD 4
============================================================

PROCEDENCIA
{descripcion_origen}
Dispositivo de evaluación: {procedencia_resultados['dispositivo_evaluacion']}
Versión de evaluación: {procedencia_resultados['version_evaluacion']}
Versión del dataset: {procedencia_resultados['version_dataset']}

SELECCIÓN
Modelo: {modelo_seleccionado}
Criterio: mayor F1 macro sobre prueba.
F1 macro: {decision_final['f1_macro']:.4f}
Aciertos: {decision_final['aciertos']} de {len(df_prueba)}

ARCHIVOS PARA EL REPORTE
resultados/comparacion_prueba.csv:
  Calidad, aciertos, tiempos y estabilidad de los tres modelos.
resultados/metricas_por_categoria.csv:
  Comparación por categoría.
resultados/confusiones_frecuentes.csv:
  Patrones de error.
resultados/modelo_N_*.csv:
  Predicciones, errores y matrices de cada modelo.
registros/registro_resultados.json:
  Relación entre modelos, archivos, tiempos y procedencia.
decision_final.json:
  Selección experimental y alcance.

INSUMOS REUTILIZABLES
dataset/dataset_prueba.csv:
  240 consultas para evaluación.
dataset/dataset_desarrollo.csv:
  60 consultas de desarrollo.
dataset/configuracion_para_prueba.json:
  Los tres modelos y la configuración fijada.
dataset/dataset_completo_trazabilidad.csv:
  Originales, traducciones y revisión.
dataset/registros_retirados.csv:
  Auditoría; no utilizar como datos de evaluación.
configuracion_modelo_seleccionado.json:
  Configuración del modelo recomendado; no incluye sus pesos.

ENTORNOS
registros/entorno_actual_notebook.json describe esta sesión.
En recuperación, el entorno de evaluación consta en
registros/registro_evaluacion_original.json.
No atribuir los tiempos anteriores al entorno actual.

CONTINUIDAD
Conservar el notebook actual con sus salidas y el primer notebook.
En recuperación, evidencias_originales/EVIDENCIAS_PRIMER_COLAB.zip
conserva también los experimentos previos de desarrollo.
En nueva evaluación, conservar por separado esos experimentos.

PENDIENTES DE LA ENTREGA
Reporte técnico, fichas de tamaño y licencia de los modelos,
y organización del repositorio con README, dependencias y código.

LIMITACIONES
Los tiempos incluyen el procesamiento del pipeline.
Los puntajes no son probabilidades calibradas.
No se validó un umbral operativo ni se midieron costos monetarios.
La recomendación requiere validación adicional antes de producción.
"""

(carpeta_respaldo / "LEEME_ENTREGA.txt").write_text(
    guia, encoding="utf-8"
)

# Inventariamos los archivos y sus huellas
inventario = []

for ruta in sorted(carpeta_respaldo.rglob("*")):
    if ruta.is_file():
        inventario.append({
            "archivo": ruta.relative_to(carpeta_respaldo).as_posix(),
            "bytes": ruta.stat().st_size,
            "sha256": hashlib.sha256(ruta.read_bytes()).hexdigest()
        })

pd.DataFrame(inventario).to_csv(
    carpeta_respaldo / "inventario_archivos.csv",
    index=False, encoding="utf-8-sig"
)

guardar_json(
    carpeta_respaldo / "manifest_entrega.json",
    {
        "fecha_respaldo_utc": marca_respaldo,
        "procedencia": procedencia_resultados,
        "archivos": {
            fila["archivo"]: {
                "bytes": fila["bytes"],
                "sha256": fila["sha256"]
            }
            for fila in inventario
        }
    }
)

# Generamos un ZIP y comprobamos su contenido
ruta_zip_entrega = Path(shutil.make_archive(
    str(carpeta_respaldo),
    "zip",
    root_dir=str(carpeta_respaldo)
))

esperados = {
    ruta.relative_to(carpeta_respaldo).as_posix()
    for ruta in carpeta_respaldo.rglob("*")
    if ruta.is_file()
}

with ZipFile(ruta_zip_entrega, "r") as paquete:
    encontrados = {
        entrada.filename
        for entrada in paquete.infolist()
        if not entrada.is_dir()
    }

    if paquete.testzip() is not None or encontrados != esperados:
        raise ValueError("Falló la comprobación del ZIP final.")

    for fila in inventario:
        huella = hashlib.sha256(
            paquete.read(fila["archivo"])
        ).hexdigest()

        if huella != fila["sha256"]:
            raise ValueError(
                f"Falló la verificación de {fila['archivo']}."
            )

print("\nProcedencia:", descripcion_origen)
print("Modelo seleccionado:", modelo_seleccionado)

print("\n📁 Carpeta del respaldo:")
print(carpeta_respaldo)
print("\n📦 ZIP para conservar en tu computadora:")
print(ruta_zip_entrega.name)

print("\n➡️ PARA EL REPORTE: resultados/")
print("➡️ PARA CONTINUAR CON LOS DATOS: dataset/")
print("➡️ PARA JUSTIFICAR LA SELECCIÓN: decision_final.json")
print("➡️ PARA CONSULTAR LA PROCEDENCIA: registros/")
print("➡️ PARA UBICAR ARCHIVOS: inventario_archivos.csv")
print("➡️ GUÍA: LEEME_ENTREGA.txt")

print("\nArchivos incluidos:", len(esperados))
print("Descarga también el notebook: Archivo → Descargar → .ipynb.")
print("El reporte y el repositorio siguen pendientes.")

print("=" * 60)
print("✅ Respaldo final creado y ZIP verificado.")
print("=" * 60)

print("\n📥 Solicitando la descarga del respaldo...")
files.download(str(ruta_zip_entrega))
print("ℹ️ Comprueba que el ZIP aparezca en tu carpeta de descargas.")

📦 Preparando el respaldo final de la comparación...

Procedencia: Resultados recuperados de la evaluación del primer Colab. Los modelos no se ejecutaron en esta sesión.
Modelo seleccionado: XLM-RoBERTa-large

📁 Carpeta del respaldo:
/content/entrega_actividad_4/comparacion_final_20261010T215423586532Z

📦 ZIP para conservar en tu computadora:
comparacion_final_20261010T215423586532Z.zip

➡️ PARA EL REPORTE: resultados/
➡️ PARA CONTINUAR CON LOS DATOS: dataset/
➡️ PARA JUSTIFICAR LA SELECCIÓN: decision_final.json
➡️ PARA CONSULTAR LA PROCEDENCIA: registros/
➡️ PARA UBICAR ARCHIVOS: inventario_archivos.csv
➡️ GUÍA: LEEME_ENTREGA.txt

Archivos incluidos: 36
Descarga también el notebook: Archivo → Descargar → .ipynb.
El reporte y el repositorio siguen pendientes.
✅ Respaldo final creado y ZIP verificado.

📥 Solicitando la descarga del respaldo...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

ℹ️ Comprueba que el ZIP aparezca en tu carpeta de descargas.
